In [1]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 141 (delta 52), reused 121 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 42.68 MiB | 26.31 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [3]:
from huggingface_hub import login
login()

In [4]:
%cd Dual_watermarking_Scheme/

/content/Dual_watermarking_Scheme


In [5]:
!ls

data  model  notebooks	README.md  requirements.txt  setup.md  src


In [9]:
!pip uninstall -y transformers peft accelerate datasets bitsandbytes

Found existing installation: transformers 5.15.0
Uninstalling transformers-5.15.0:
  Successfully uninstalled transformers-5.15.0
Found existing installation: peft 0.20.0
Uninstalling peft-0.20.0:
  Successfully uninstalled peft-0.20.0
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: datasets 5.0.1
Uninstalling datasets-5.0.1:
  Successfully uninstalled datasets-5.0.1
Found existing installation: bitsandbytes 0.50.1
Uninstalling bitsandbytes-0.50.1:
  Successfully uninstalled bitsandbytes-0.50.1


In [10]:
!pip install -U \
transformers==5.15.0 \
peft==0.20.0 \
accelerate==1.14.0 \
datasets==5.0.1 \
bitsandbytes==0.50.1

  Using cached transformers-5.15.0-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached bitsandbytes-0.50.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached transformers-5.15.0-py3-none-any.whl (11.7 MB)
Using cached peft-0.20.0-py3-none-any.whl (775 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 12.8 MB/s eta 0:00:00
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached bitsandbytes-0.50.1-py3-none-manylinux_2_24_x86_64.whl (41.0 MB)


In [ ]:
import bitsandbytes,transformers,peft,accelerate

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

ModuleNotFoundError: No module named 'transformesr'

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [14]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("BitsAndBytesConfig created successfully:")
print(bnb_config)

# Quick check that bitsandbytes CUDA ops are reachable
import bitsandbytes as bnb
print("\nbitsandbytes CUDA setup check:")
print(bnb.__version__)

BitsAndBytesConfig created successfully:
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


bitsandbytes CUDA setup check:
0.50.1


In [15]:
import transformers, peft, accelerate, datasets, bitsandbytes

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

In [16]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import PeftModel


BASE_MODEL = "facebook/opt-2.7b"
adapter_path = "./model/v1"


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)


model = PeftModel.from_pretrained(
    model,
    adapter_path
)

model.eval()


prompt = """
Topic: Probability
Difficulty: Hard
Question:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )


print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at './model/v1'